# FoodLens 成员 C：16 图 Schema 学习验证

仅在 Colab NVIDIA GPU 运行。先用独立目录从预训练模型新建 LoRA，训练 25 步；人工检查曲线和原始输出后才可选择继续至总计 100 步。不恢复原来的两步 smoke 断点，不自动启动完整训练。

沿用已审核的 ZIP 数据准备、固定模型版本、CUDA 依赖保护和完整源码 SHA 核对。test 只做静态数据完整性检查，不进入训练或实验评价。所有图表和样例来自运行时生成的文件，本 Notebook 不附带实验成绩。


In [ ]:
import os, sys, json, subprocess, platform, uuid
from pathlib import Path
from datetime import datetime, timezone

if not Path("/content").is_dir() or "google.colab" not in sys.modules and not os.environ.get("COLAB_RELEASE_TAG"):
    try:
        import google.colab
    except ImportError as exc:
        raise RuntimeError("请在 Google Colab 托管 GPU 运行时运行；不支持本地训练") from exc
import torch
assert torch.cuda.is_available(), "请先选择 NVIDIA GPU 运行时；这里不会退回 CPU 训练"
print(platform.python_version(), torch.__version__, torch.cuda.get_device_name(0))


## 获取成员 C 分支代码
先在推送后用 `git rev-parse HEAD` 取得最终完整 40 位 SHA，填写下方 EXPECTED_CODE_COMMIT（不能填旧的 6a8ac50）。留空、缩写或实际分支 HEAD 不一致都会停止；不会自动 checkout 其他版本或 main。已有目录只读检查，不自动拉取、切换、覆盖或清理。若目录版本旧，请在新运行时重试，或自行处理目录。所有命令均检查退出状态。


In [ ]:
BRANCH = "feat/member-c-diet-record"
REPO_URL = "https://github.com/liuxy2006/FoodLens.git"
REPO = Path("/content/FoodLens-member-c")
EXPECTED_CODE_COMMIT = ""  # 必填：推送本次最终提交后，粘贴 git rev-parse HEAD 的完整 40 位 SHA
import re
if not re.fullmatch(r"[0-9a-fA-F]{40}", EXPECTED_CODE_COMMIT):
    raise ValueError("请填写审核并推送后的最终完整 40 位 commit SHA；不要填写旧提交或短 SHA")
EXPECTED_CODE_COMMIT = EXPECTED_CODE_COMMIT.lower()
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "branch", "--show-current"], text=True).strip() == BRANCH
assert subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip() == REPO_URL
assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "代码目录有修改，先人工检查"
for relative in ["task2_diet_record/checkpoint.py", "task2_diet_record/integration.py", "colab/member_c_train.ipynb", "task2_diet_record/prepare_colab_data.py", "task2_diet_record/colab_dependencies.py", "task2_diet_record/schema_experiment.py", "task2_diet_record/overfit_config.json"]:
    assert (REPO / relative).is_file(), "远程分支尚无第二阶段代码；请审核后自行推送该分支，再开新运行时"
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert CODE_COMMIT == EXPECTED_CODE_COMMIT, "代码版本与预期不一致"
os.chdir(REPO)
print("branch:", BRANCH, "commit:", CODE_COMMIT)
def run_module(module, *arguments):
    subprocess.run([sys.executable, "-m", module, *map(str, arguments)], cwd=REPO, check=True)


## 安装依赖
保留 Colab 预装的 CUDA PyTorch：安装前后在新进程读取 torch.__version__、torch.version.cuda、torch.cuda.is_available()；用精确约束锁定已安装的 torch/torchvision/torchaudio/triton，并先审查 pip dry-run 清单。若计划替换 GPU 运行时、安装 Windows/XPU 依赖，或安装后 CUDA 不可用，就停止。固定现有框架依赖，记录实际环境。若安装后提示需要重启，请重启会话后从头执行；已满足的依赖不会重复安装。


In [ ]:
# 仅在 Colab 执行：安装模块将保存 before/after、约束和 pip 计划。
INSTALL_REPORT = Path("/content/foodlens_install_" + uuid.uuid4().hex[:8])
run_module("task2_diet_record.colab_dependencies", "--output", INSTALL_REPORT)


## 挂载 Drive、设置路径和记录环境
数据只需以两个 ZIP 上传到 Drive；下一格自动识别 ZIP 层级、复制并解压到 Colab 本地。已有本地完整数据校验通过后直接复用；不完整/不一致数据会明确报错，不自动覆盖或清理。输出目录使用唯一运行 ID，断线后的恢复必须指定旧断点路径，不能覆盖旧实验。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FoodLens")
DATASET_DIR = DRIVE_ROOT / "dataset"
DATA_ROOT = Path("/content/foodlens_data")
ARCHIVE_CACHE = Path("/content/foodlens_archives")
REVISION = "7e3e67edbbed1bf9888184d9df282b700a323964"
MODEL_DIR = DRIVE_ROOT / "models" / ("SmolVLM-256M-Instruct-" + REVISION[:8])
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
RUN_ROOT = DRIVE_ROOT / "outputs" / ("schema_overfit_" + RUN_ID)
RUN_ROOT.mkdir(parents=True, exist_ok=False)
os.environ["HF_HOME"] = str(DRIVE_ROOT / "cache" / "huggingface")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import shutil
shutil.copytree(INSTALL_REPORT, RUN_ROOT / "dependency_install")
(RUN_ROOT / "source.json").write_text(json.dumps({"branch": BRANCH, "commit": CODE_COMMIT, "model_revision": REVISION}, indent=2))
(RUN_ROOT / "pip_freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
(RUN_ROOT / "nvidia_smi.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
run_module("task2_diet_record.device_probe", "--output", RUN_ROOT / "device_probe.json")
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
print("所有结果保存至:", RUN_ROOT)


In [ ]:
from task2_diet_record.prepare_colab_data import prepare_data
preparation = prepare_data(DATASET_DIR, DATA_ROOT, ARCHIVE_CACHE)
assert Path(preparation["data_root"]) == DATA_ROOT
print(json.dumps(preparation, ensure_ascii=False, indent=2))
(RUN_ROOT / "data_preparation.json").write_text(json.dumps(preparation, indent=2))
# 已核对全部 train/val/test 数量与图片引用；再对训练/验证图片做少量实际解码。
from task2_diet_record.data import DietDataset, read_annotations
for split in ("train", "val"):
    dataset = DietDataset(DATA_ROOT / "raw/annotations" / (split + ".jsonl"), DATA_ROOT, split)
    for index in range(min(8, len(dataset))):
        dataset[index]["image"].close()
    print(split, "前 8 张图片加载成功")
# test 在这里仅作静态完整性核对，标签不用于训练、选模、调参或提示词优化。
val_rows = read_annotations(DATA_ROOT / "raw/annotations/val.jsonl")[:2]
VAL_SMOKE = RUN_ROOT / "val_smoke.jsonl"
VAL_SMOKE.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in val_rows), encoding="utf-8")


## 核实并下载唯一指定模型
已查询官方仓库：必要文件约 516.6 MB，其中 safetensors 约 513.0 MB。先输出清单，再下载相同 revision 的模型与 Processor；复跑时缓存可复用。下载后核对大小和 LFS SHA256，不下载 Qwen/ONNX 等其他权重。


In [ ]:
run_module("task2_diet_record.prepare_model", "--revision", REVISION, "--destination", MODEL_DIR,
           "--manifest", RUN_ROOT / "model_manifest.json")
run_module("task2_diet_record.prepare_model", "--revision", REVISION, "--destination", MODEL_DIR,
           "--manifest", RUN_ROOT / "model_manifest.json", "--download")


## 先做 25 步检查
固定 train 的前 16 张、val 的前 4 张；batch 1、累积 1、rank 8、alpha 16、lr 1e-4。7 epochs 最多提供 112 步机会，但入口在 25 步停止，后续上限是 100，不能靠修改 epoch 突破上限。
步骤 0 和 25 对固定的 16+4 张图片进行生成评价；逐步 training loss 保存在 steps.jsonl。验证损失使用固定 4 张 val，不能当作完整验证集成绩。
首次运行 OVERFIT_RESUME 留空，断线恢复只允许本实验的完整断点，并必须仍少于 25 步。


In [ ]:
OVERFIT_RESUME = ""
STAGE25 = RUN_ROOT / "schema_step25"
arguments = ["--model-path", MODEL_DIR, "--revision", REVISION, "--data-root", DATA_ROOT,
             "--stage", 25, "--output", STAGE25]
if OVERFIT_RESUME:
    arguments += ["--resume", OVERFIT_RESUME]
run_module("task2_diet_record.schema_experiment", *arguments)
print((STAGE25 / "decision.json").read_text())


## 查看曲线和原始输出
PNG 包含每步 training loss、validation loss、JSON/Schema 合法率、F1、有效覆盖率和五项营养 MSE。无合法预测时的有效子集指标保持缺失；全零补值 MSE 单独标明。训练子集分数只表示记忆能力。
报告保存全部 16+4 张图片的原始输出、Schema 错误、错误字段计数、token 上限信息和真值对照；真值仅在生成完成后加入报告，不传给模型。


In [ ]:
run_module("task2_diet_record.learning_curves", "--runs", STAGE25, "--output", RUN_ROOT / "curves25")
from IPython.display import display, Image

display(Image(filename=str(RUN_ROOT / "curves25/learning_curves.png")))
report25 = json.loads((STAGE25 / "monitor/step-000025.json").read_text())
for split in ("train", "val"):
    report = report25[split]
    print(split, report["scope"], "schema rate:", report["metrics"]["schema_valid_rate"],
          "unexpected keys:", report["unexpected_key_occurrences"])
    for sample in report["predictions"][:2]:
        print(sample["dish_id"], sample["prediction"], sample["schema"], sample.get("generation"))
print("监督文本核对见 schema_step25/batch_inspection_epoch_*.json；不要仅凭 Loss 下降作判断。")


## 可选：审查后继续至总计 100 步
默认关闭。查看 25 步原文、监督文本、截断状态及所有指标后，填写审查说明并显式打开开关。新建输出目录，恢复本实验的 checkpoint，不覆盖 25 步或此前 smoke 结果。
若 100 步时训练子集 Schema 合法率仍为 0，停止增加步数，先诊断提示、监督 token、EOS/生成长度和注入模块。


In [ ]:
CONTINUE_TO_100 = False
REVIEW_NOTE = ""
STAGE100 = RUN_ROOT / "schema_step100"
if CONTINUE_TO_100:
    assert REVIEW_NOTE.strip(), "先查看 25 步报告，再填写审查说明"
    (RUN_ROOT / "review_before_100.txt").write_text(REVIEW_NOTE, encoding="utf-8")
    checkpoint = json.loads((STAGE25 / "last_checkpoint.json").read_text())["path"]
    run_module("task2_diet_record.schema_experiment", "--model-path", MODEL_DIR, "--revision", REVISION,
               "--data-root", DATA_ROOT, "--stage", 100, "--resume", checkpoint, "--output", STAGE100)
    print((STAGE100 / "decision.json").read_text())
    run_module("task2_diet_record.learning_curves", "--runs", STAGE25, STAGE100, "--output", RUN_ROOT / "curves100")
    display(Image(filename=str(RUN_ROOT / "curves100/learning_curves.png")))
else:
    print("未启动 100 步阶段。先审查 25 步结果。")


## 后续 1 Epoch 探索配置（默认关闭）
只有训练子集 Schema 接近全对、验证子集开始出现合法输出、梯度/监督核对均正常，且你审查过 F1、MSE、截断和 T4 多步资源记录后，才考虑正式探索。下面的门槛只是防误操作，不是泛化性能验收标准。
从原始预训练模型新建 LoRA；不把 16 图记忆结果当作正式初始化。完整 train=2338 / val=414，test 不参与；只运行 1 epoch。
暂定 batch 1、累积 8、checkpoint 间隔 50、验证间隔 100，需根据实际连续多步、读图、验证和保存开销调整，不从单步时间推算全程。


In [ ]:
RUN_FULL_TRAINING = False
SCHEMA_LEARNING_REVIEWED = False
FULL_RESUME_CHECKPOINT = ""
if RUN_FULL_TRAINING:
    assert SCHEMA_LEARNING_REVIEWED, "必须先人工审查格式学习和多步资源报告"
    latest = STAGE100 if CONTINUE_TO_100 else STAGE25
    snapshots = sorted((latest / "monitor").glob("step-*.json"))
    evidence = json.loads(snapshots[-1].read_text())
    assert evidence["train"]["metrics"]["schema_valid_rate"] >= 0.9, "训练子集格式仍不稳定"
    assert evidence["val"]["metrics"]["schema_valid_rate"] > 0, "先诊断验证输出"
    full_args = ["--config", REPO / "task2_diet_record/exploratory_config.json", "--model-path", MODEL_DIR,
                 "--revision", REVISION, "--data-root", DATA_ROOT, "--device", "cuda",
                 "--eval-every", 100, "--generate-train", 16, "--generate-val", 4,
                 "--checkpoint-every", 50, "--output", RUN_ROOT / "full_one_epoch"]
    if FULL_RESUME_CHECKPOINT:
        full_args += ["--resume", FULL_RESUME_CHECKPOINT]
    run_module("task2_diet_record.train", *full_args)
else:
    print("未启动完整训练。")
